# ICT-4442 Mini Project: Attention-based Tabular Autoencoder
## Comparative Evaluation of Deep Learning Architectures for Credit Card Fraud Detection

**Student:** Priyansh Nandan (230953450)  
**Role:** Common Preprocessing Pipeline & Attention Autoencoder Architecture

This notebook demonstrates the end-to-end execution of:
1. Dataset loading and schema validation
2. Strict chronological splitting (70% train, 15% validation, 15% test)
3. Training-only feature scaling (zero leakage)
4. Legitimate-only training subset extraction
5. Attention Autoencoder architecture (Feature Tokenizer -> 2-layer Transformer -> 8-dim bottleneck -> Decoder)
6. Validation threshold selection and held-out test evaluation

In [ ]:
import sys
from pathlib import Path

# Ensure project root is in python path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import numpy as np
import pandas as pd
from src.config import config, RAW_DATASET_PATH
from src.preprocessing import check_dataset_exists, load_raw_data, inspect_data, chronological_split, prepare_features_and_targets, scale_features, get_autoencoder_training_subset
from src.models.attention_autoencoder import AttentionAutoencoder
from src.evaluation.evaluate_autoencoder import find_optimal_threshold, compute_comprehensive_metrics

print(f"PyTorch version: {torch.__version__}")
print(f"Expected dataset location: {RAW_DATASET_PATH}")
print(f"Dataset exists locally: {check_dataset_exists(RAW_DATASET_PATH)}")

## 1. Inspecting the Dataset
When `data/raw/creditcard.csv` is present, we load and inspect class distributions, missing values, duplicates, and transaction time ranges.

In [ ]:
if check_dataset_exists(RAW_DATASET_PATH):
    df_raw = load_raw_data(RAW_DATASET_PATH)
    summary = inspect_data(df_raw)
else:
    print("Raw dataset not found at data/raw/creditcard.csv.")
    print("Please download from https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud and place in data/raw/")

## 2. Attention Autoencoder Model Demonstration
Here we verify the architectural specifications from the Phase 1 synopsis:
- 2 Transformer encoder layers
- 4 attention heads
- 8-unit bottleneck
- 30-dimensional input and output

In [ ]:
model = AttentionAutoencoder(config.model)
print(model)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal trainable parameters: {total_params:,}")

# Forward pass with dummy batch of shape (batch_size=4, num_features=30)
dummy_x = torch.randn(4, 30)
reconstruction, latent = model(dummy_x)
reconstruction_error = model.compute_reconstruction_error(dummy_x)

print(f"Input shape:                {dummy_x.shape}")
print(f"Bottleneck shape:           {latent.shape} (strictly 8 dimensions)")
print(f"Reconstructed shape:        {reconstruction.shape} (reconstructs 30 features)")
print(f"Reconstruction MSE scores:  {reconstruction_error.detach().numpy()}")